# SH Solver vs HEOM Benchmark

This notebook compares the **SH solver**, implementing the
Stochastic Pseudomode Model from
[PRX Quantum **4**, 030316 (2023)](https://doi.org/10.1103/PRXQuantum.4.030316),
against the **Hierarchical Equations of Motion (HEOM)** reference for a two-level
system coupled to a classical Gaussian bath.

**Physical setup:**
- Two-level system with Hamiltonian $H_S = \frac{\Omega}{2}\,\sigma_x$
- Constant classical bath correlation $C_\text{class}(t) = \sigma^2$
- Coupling operator $\hat{s} = \sigma_z$

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from qutip import (
    basis, spre, spost, expect,
    ExponentialBosonicEnvironment,
)
from qutip.solver.heom import HEOMSolver

# Ensure the solver module is importable
sys.path.insert(0, str(Path("../sumo/Classes").resolve()))
from shsolver import SHSolver

## 2. Physical Parameters

In [ ]:
Omega = 1
sigma = 0.05 * Omega
T     = 40 / Omega

## 3. Numerical Parameters

In [ ]:
N_corr  = 600
n_cut   = 600
n_noise = 500

## 4. Operator Construction

In [ ]:
sigmaz = basis(2, 0) * basis(2, 0).dag() - basis(2, 1) * basis(2, 1).dag()
sigmax = basis(2, 1) * basis(2, 0).dag() + basis(2, 0) * basis(2, 1).dag()

H_S  = Omega / 2.0 * sigmax
psi0 = basis(2, 0)

L    = -1j * (spre(H_S) - spost(H_S))
H_xi = -1j * (spre(sigmaz) - spost(sigmaz))

## 5. Time Grids and Correlation Function

In [ ]:
t_corr_list     = np.linspace(-T, T, 2 * N_corr + 1)
t_dynamics_list = np.linspace(0, T, N_corr)

c_corr_list = np.full_like(t_corr_list, sigma**2)

## 6. Run the Stochastic Hamiltonian Solver

In [ ]:
solver = SHSolver()

coeff_list = solver.compute_coefficients_basis(t_corr_list, c_corr_list, n_cut)
xi_list    = solver.generate_fields(t_corr_list, coeff_list, n_cut, n_noise)

dynamics_average, sigma_dynamics, dynamics_list, states_list = (
    solver.average_dynamics_parallel(
        L=L, H_xi=H_xi, xi_list=xi_list,
        c_list=[],
        t_list=t_dynamics_list, t_corr_list=t_corr_list, psi0=psi0,
        n_noise=n_noise, obs_list=[sigmaz],
        args={},
        options={
            "atol": 1e-12,
            "rtol": 1e-10,
            "nsteps": 200000,
            "store_states": True,
        },
    )
)

## 7. Run the HEOM Reference

In [ ]:
env = ExponentialBosonicEnvironment([sigma**2], [0.0], [], [])

heom_solver = HEOMSolver(H_S, (env, sigmaz), max_depth=50)
result_heom = heom_solver.run(psi0 * psi0.dag(), t_dynamics_list)

## 8. Transform to the Lab Frame

Both solvers work in the interaction picture.

In [ ]:
def to_lab_frame(states, H_S, t_list, observable):
    """Rotate states to the lab frame and return expectation values."""
    rotated = [
        (1j * H_S * t).expm() * rho * (-1j * H_S * t).expm()
        for rho, t in zip(states, t_list)
    ]
    return np.real(expect(observable, rotated))

sz_heom = to_lab_frame(result_heom.states, H_S, t_dynamics_list, sigmaz)

In [ ]:
states_arr = np.array([[rho.full() for rho in traj] for traj in states_list])
sz_matrix  = sigmaz.full()
unitaries  = [(1j * H_S * t).expm().full() for t in t_dynamics_list]

sz_all = np.array([
    [np.real(np.trace(sz_matrix @ (U @ states_arr[k, t] @ U.conj().T)))
     for t, U in enumerate(unitaries)]
    for k in range(n_noise)
])

sz_mean = sz_all.mean(axis=0)
sz_sem  = sz_all.std(axis=0) / np.sqrt(n_noise)

## 9. Comparison Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))

ax.plot(t_dynamics_list, sz_heom, linewidth=2.5, label="HEOM", color="tab:blue")
ax.plot(t_dynamics_list, sz_mean, linewidth=2.5, label="SPM", color="tab:orange",
        linestyle="--")
ax.fill_between(
    t_dynamics_list, sz_mean - sz_sem, sz_mean + sz_sem,
    alpha=0.3, color="tab:orange", label=r"SPM $\pm$ SEM",
)

ax.set_xlabel(r"Time $t\;(1/\Omega)$", fontsize=12)
ax.set_ylabel(r"$\langle \hat{\sigma}_z \rangle$", fontsize=12)
ax.legend(frameon=False, fontsize=10)
ax.set_title("Stochastic Pseudomode Model vs HEOM", fontsize=13)
fig.tight_layout()
plt.show()